# wsi-metastasis-seg — training on Colab

Trains from an **exported patch set** (~4 GB) instead of the 230 GB slide
archive. See `docs/COLAB.md` for what that trades away.

Before running: set **Runtime → Change runtime type → GPU**.


## 1. Confirm there is a GPU

Stop here if this fails. Everything below assumes CUDA.


In [ ]:
!nvidia-smi
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU. Runtime -> Change runtime type -> GPU'


## 2. Code

Cloned, not uploaded, so the notebook and the repository cannot drift.


In [ ]:
# Always start from /content. If a previous run left the kernel inside
# the repo directory and that directory was then deleted, every later
# command fails with 'getcwd: cannot access parent directories'. An
# absolute chdir recovers from that state; a relative one cannot.
%cd /content

REPO = 'https://github.com/arcii99/medical-image-segmentation-wsi.git'  # <-- edit

# Re-runnable: git clone refuses a non-empty target, so clear it first.
!rm -rf /content/wsi-metastasis-seg
!git clone -q $REPO /content/wsi-metastasis-seg
%cd /content/wsi-metastasis-seg
!pip install -q -e '.[dev]' segmentation-models-pytorch

# Two separate questions: is the tree the version you expect, and would a
# clone of it even be complete? A .gitignore rule once excluded src/data/
# from every commit, and only the version check caught it.
!python scripts/version.py
!python scripts/check_repo.py || true


## 3. Data

Extract shards to **/content** (local disk). Reading 44,000 small files
over the Drive mount makes the GPU wait on IO all session.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/wsi'   # <-- edit if different


In [ ]:
!mkdir -p /content/patchset/files
!cp $DRIVE/patchset/manifest.parquet $DRIVE/patchset/export_info.json /content/patchset/
!for t in $DRIVE/patchset/shards/*.tar; do tar xf "$t" -C /content/patchset/files; done
!echo "extracted $(ls /content/patchset/files | wc -l) files"
!cat /content/patchset/export_info.json


In [ ]:
# sanity-check one sample before spending GPU time on 44,000 of them
import sys; sys.path.insert(0, '/content/wsi-metastasis-seg')
import numpy as np
from src.data.patchset import PatchSetDataset
ds = PatchSetDataset('/content/patchset', 'train', train=True)
s = ds[0]
print('image', s['image'].shape, s['image'].dtype)
print('mask ', s['mask'].shape, 'unique', np.unique(s['mask']))
print('tumor patches', int((ds.index.tumor_frac > 0.05).sum()), 'of', len(ds))


## 4a. Dry run (seconds)

One forward+backward, then exit. Confirms shapes, finite loss, flowing
gradients and a binary mask before anything long starts.


In [ ]:
!python scripts/03_train.py \
  --config configs/base.yaml configs/data_camelyon16.yaml \
           configs/model_unet_effb0.yaml configs/patchset.yaml \
  --set data.patchset_root=/content/patchset --dry-run


## 4. Wiring check first (gate V6.2)

Overfit a single batch. About two minutes on a T4.

**Expect `loss <= 0.05` and `dice >= 0.97`.** A model that cannot memorise
one batch has a defect no amount of data will fix, and finding it now
costs two minutes instead of a session.


In [ ]:
!python scripts/03_train.py \
  --config configs/base.yaml configs/data_camelyon16.yaml \
           configs/model_unet_effb0.yaml configs/patchset.yaml \
  --set data.patchset_root=/content/patchset \
  --set train.overfit_batches=1 --set train.max_steps=300 \
  --set data.augment=false --set paths.ckpt=/content/smoke


## 5. Train

Checkpoints go to **Drive** so a disconnect is survivable.

~20 min/epoch at 44k patches on a T4, so 40 epochs needs 2–3 sessions.


In [ ]:
!python scripts/03_train.py \
  --config configs/base.yaml configs/data_camelyon16.yaml \
           configs/model_unet_effb0.yaml configs/patchset.yaml \
  --set data.patchset_root=/content/patchset \
  --set paths.ckpt=$DRIVE/ckpt \
  --set train.max_epochs=40


### Resuming after a disconnect

Re-run cells 1–3, then this. Check the loss continues rather than jumping.


In [ ]:
RUN_ID = ''   # <-- the directory name under $DRIVE/ckpt
!python scripts/03_train.py \
  --config configs/base.yaml configs/data_camelyon16.yaml \
           configs/model_unet_effb0.yaml configs/patchset.yaml \
  --set data.patchset_root=/content/patchset \
  --set paths.ckpt=$DRIVE/ckpt \
  --resume $DRIVE/ckpt/$RUN_ID/last.pt


### Checkpoint completeness (gate V7.2)


In [ ]:
import torch
c = torch.load(f'{DRIVE}/ckpt/{RUN_ID}/best.pt', map_location='cpu')
req = {'state_dict','ema_state_dict','optimizer','epoch','threshold',
       'resolved_config','git_sha','dirty','pip_freeze','index_hash','seed'}
missing = req - set(c)
print('MISSING', missing) if missing else print('COMPLETE')
print('threshold', c['threshold'], '| epoch', c['epoch'], '| dirty', c['dirty'])
assert 0.30 <= c['threshold'] <= 0.70, 'tau outside the expected band (ADR-006)'


## 6. Inference on the test slides

Whole slides are needed here — but they are **downloaded from S3 inside
the notebook**, which is far faster than uploading. About 16 GB.


In [ ]:
!tar xzf $DRIVE/meta.tar.gz -C /content/
import pandas as pd
sl = pd.read_parquet('/content/artifacts/index/slides.parquet')
test = sl[sl.split == 'test'].slide_id.tolist()
print(len(test), 'test slides:', test)


In [ ]:
!pip install -q awscli
!mkdir -p /content/slides
for s in test:
    !aws s3 cp --no-sign-request --quiet s3://camelyon-dataset/CAMELYON16/images/{s}.tif /content/slides/{s}.tif
    !aws s3 cp --no-sign-request --quiet s3://camelyon-dataset/CAMELYON16/annotations/{s}.xml /content/slides/{s}.xml || true
!du -sh /content/slides


In [ ]:
CKPT = f'{DRIVE}/ckpt/{RUN_ID}/best.pt'
for s in test:
    !python scripts/04_infer_slide.py --ckpt $CKPT \
      --slide /content/slides/{s}.tif \
      --set paths.tissue=/content/artifacts/tissue \
      --set paths.artifacts=/content/artifacts


## 7. Evaluate

FROC is the primary metric. With 9 test slides and ~48 evaluable lesions,
each lesion is ~2% of sensitivity — report intervals, not point estimates.


In [ ]:
!python scripts/05_evaluate.py --run-id $RUN_ID --split test \
  --set paths.artifacts=/content/artifacts
import json
print(json.dumps(json.load(open(f'/content/artifacts/reports/{RUN_ID}/metrics.json')), indent=2))


## 8. Copy results back


In [ ]:
!mkdir -p $DRIVE/results
!cp -r /content/artifacts/reports $DRIVE/results/
!cp -r /content/artifacts/heatmaps $DRIVE/results/
!cp -r /content/artifacts/overlays $DRIVE/results/
!du -sh $DRIVE/results


---
**When writing this up**, state that the model was trained on an exported
patch set: no per-epoch coordinate jitter and no hard-negative mining.
That is a real difference from the documented procedure, and a reader
comparing against published CAMELYON16 numbers should know.
